# Lesson 03b — ChromaDB + LangChain RAG Pipeline
**Domain:** Scientific Literature  
**Dataset:** arXiv abstracts (HF) + `pubmed_qa` labeled (HF)  
**Time estimate:** ~2 h

## Learning objectives
- ChromaDB: collection, add, query, metadata filtering
- LangChain `create_retrieval_chain` with ChromaDB retriever
- Prompt template with retrieved context
- Hallucination check: does the answer use the context?

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from openai import OpenAI
from llm_checker import check, hint, evaluate, progress

local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODEL = "local-model"

def llm(messages, max_tokens=300, temperature=0.1):
    r = local_client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=temperature
    )
    return r.choices[0].message.content.strip()

print("✅ LM Studio client ready")

## Load arXiv + PubMed datasets

In [ ]:
from datasets import load_dataset

# arXiv abstracts (re-use from 03a)
print("Loading arXiv abstracts…")
arxiv_ds = load_dataset("gfissore/arxiv-abstracts-2021", split="train").select(range(2000))
abstracts = [
    {
        "id": str(i),
        "title": row.get("title", f"Abstract {i}"),
        "text":  row.get("abstract", "") or row.get("text", ""),
        "category": str(row.get("categories", "cs"))[:20],
        "year": "2021",
    }
    for i, row in enumerate(arxiv_ds)
]
print(f"✅ {len(abstracts)} arXiv abstracts loaded")

# PubMed QA (for evaluation questions)
print("Loading PubMed QA…")
try:
    pubmed_ds  = load_dataset("pubmed_qa", "pqa_labeled", split="train").select(range(50))
    pubmed_qas = [{"question": row["question"], "answer": row.get("long_answer","")} for row in pubmed_ds]
    print(f"✅ {len(pubmed_qas)} PubMed QA pairs loaded")
except Exception as e:
    print(f"⚠️  PubMed QA not available ({e}). Using fallback questions.")
    pubmed_qas = [
        {"question": "What are the effects of transformer attention mechanisms on language modeling?", "answer": ""},
        {"question": "How does federated learning maintain privacy?", "answer": ""},
        {"question": "What is the role of BERT in NLP tasks?", "answer": ""},
        {"question": "How do graph neural networks work?", "answer": ""},
        {"question": "What are diffusion models used for?", "answer": ""},
    ] * 10

---
## 🔵 EXAMPLE — Ex 03b-1: Build a ChromaDB collection

Index 200 arXiv abstracts with metadata. Run 3 queries, print top-3 results.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Use sentence-transformers embedding function (local, no API)
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("arxiv")
except Exception:
    pass

collection = chroma_client.create_collection("arxiv", embedding_function=ef)

# Index 200 abstracts
batch_size = 50
for i in range(0, 200, batch_size):
    batch = abstracts[i:i+batch_size]
    collection.add(
        documents=[a["text"] for a in batch],
        metadatas=[{"title": a["title"], "category": a["category"], "year": a["year"]} for a in batch],
        ids=[a["id"] for a in batch],
    )
    print(f"  Indexed {min(i+batch_size, 200)}/200 abstracts")

print(f"\n✅ Collection size: {collection.count()}")

# Run 3 queries
test_queries = [
    "attention mechanism in transformers",
    "protein structure prediction deep learning",
    "federated learning privacy",
]

for query in test_queries:
    results = collection.query(query_texts=[query], n_results=3, include=["documents","metadatas","distances"])
    print(f"\nQuery: {query!r}")
    for j in range(3):
        title = results["metadatas"][0][j]["title"]
        dist  = results["distances"][0][j]
        print(f"  {j+1}. [{dist:.3f}] {title[:70]}")

check([
    (collection.count() >= 200,  f"Collection has ≥ 200 documents ({collection.count()})"),
    (True,                        "3 queries returned results with metadata"),
], exercise_id="03b-1")

---
## 🟡 EXERCISE — Ex 03b-2: Full RAG Q&A pipeline

Build `rag_answer(question: str) -> dict` with keys: `answer`, `sources`, `context_used`.
Use the ChromaDB collection from 03b-1. Test on 10 science questions.

**Auto-check verifies:**
- Returns dict with all 3 keys
- `sources` is a non-empty list of strings
- `answer` differs from the question

In [ ]:
SYSTEM_RAG = """You are a scientific assistant. Answer the question based ONLY on the provided context.
If you cannot find the answer in the context, say: "I don't know based on the available papers."
Be concise (2-4 sentences).""".strip()

def rag_answer(question: str) -> dict:
    """
    Returns {
        "answer": str,
        "sources": list[str],    # list of paper titles
        "context_used": str,     # concatenated retrieved context
    }
    """
    # TODO: implement
    # 1. Query ChromaDB for top-4 relevant abstracts
    # 2. Build context string from retrieved documents
    # 3. Call LLM with system prompt + context + question
    # 4. Return dict with answer, sources (titles), context_used
    raise NotImplementedError("Implement rag_answer()")

In [ ]:
test_questions_10 = [
    "How do transformer models handle long sequences?",
    "What is self-supervised learning?",
    "How are graph neural networks used for molecular property prediction?",
    "What is contrastive learning and how is it used?",
    "How does BERT pre-training work?",
    "What are the main approaches to few-shot learning?",
    "How do generative adversarial networks generate images?",
    "What is meta-learning?",
    "How is reinforcement learning from human feedback used?",
    "What are mixture-of-experts models?",
]

try:
    rag_results = [rag_answer(q) for q in test_questions_10]

    all_have_answer   = all("answer"       in r for r in rag_results)
    all_have_sources  = all("sources"      in r and len(r["sources"]) > 0 for r in rag_results)
    all_have_context  = all("context_used" in r for r in rag_results)
    answer_not_question = all(r["answer"] != q for r, q in zip(rag_results, test_questions_10))

    for i, (q, r) in enumerate(zip(test_questions_10, rag_results)):
        print(f"Q{i+1}: {q[:60]}")
        print(f"  A: {r['answer'][:120]}")
        print(f"  Sources: {r['sources'][:2]}")
        print()

    check([
        (all_have_answer,     "All results have 'answer' key"),
        (all_have_sources,    "All results have non-empty 'sources'"),
        (all_have_context,    "All results have 'context_used' key"),
        (answer_not_question, "Answer differs from question"),
    ], exercise_id="03b-2")

except NotImplementedError:
    print("⚠️  Implement rag_answer() first.")

---
## 🔴 CHALLENGE — Ex 03b-3: Out-of-corpus hallucination test

Formulate 5 questions with no answer in the arXiv corpus.
Add to system prompt: "If you cannot find the answer, say: I don't know."
Evaluate: does the LLM comply?

In [ ]:
# 5 questions clearly outside the arXiv corpus (2021 CS papers)
oos_questions = [
    "Who won the FIFA World Cup in 2022?",
    "What is the population of Warsaw, Poland?",
    "When was the Eiffel Tower built?",
    "What is the recipe for traditional Polish bigos?",
    "Who directed the movie Inception?",
]

print("Testing hallucination behaviour on out-of-corpus questions:\n")
oos_results = []
for q in oos_questions:
    try:
        r = rag_answer(q)
        knows_answer = ("don't know" in r["answer"].lower() or "cannot" in r["answer"].lower()
                        or "not in" in r["answer"].lower() or "no information" in r["answer"].lower())
        oos_results.append({"question": q, "answer": r["answer"], "admitted_unknown": knows_answer})
        verdict = "✅ admitted" if knows_answer else "⚠️  hallucinated"
        print(f"{verdict}: {q}")
        print(f"         {r['answer'][:100]}\n")
    except NotImplementedError:
        print("⚠️  Implement rag_answer() first.")
        break

if oos_results:
    n_admitted = sum(1 for r in oos_results if r["admitted_unknown"])
    print(f"LLM admitted 'I don't know': {n_admitted}/5")
    print("# Comment on remaining cases: subtle hallucinations usually involve context mismatch")

    check([
        (len(oos_results) == 5, "5 out-of-corpus questions tested"),
        (n_admitted >= 3,       "≥ 3 of 5: LLM says it doesn't know"),
    ], exercise_id="03b-3")

In [ ]:
progress()

---
## Readiness checklist before moving to 03c
- [ ] ChromaDB collection indexed with ≥ 200 arXiv abstracts
- [ ] RAG Q&A pipeline works end-to-end (question → answer + sources)
- [ ] Hallucination test run on 5 out-of-corpus questions